# [문제 1] Fashion MNIST 데이터 정규화를 위한 Mean과 Std. 값 찾기

In [14]:
import torch
from torch.utils.data import random_split, DataLoader
from torchvision import datasets, transforms
from wandb.util import download_file_into_memory

In [25]:
data_path = "."
f_mnist_train = datasets.FashionMNIST(data_path, train=True, download=True, transform=transforms.ToTensor())
f_mnist_train, f_mnist_validation = random_split(f_mnist_train, [55_000, 5_000])

images = torch.stack([img for img, _ in f_mnist_train], dim=0)

mean = images.mean()
std = images.std()

print(f"Mean: {mean.item():.4f}")
print(f"Std: {std.item():.4f}")

Mean: 0.2860
Std: 0.3530


# [문제 2] Fashion MNIST 데이터에 대하여 CNN 학습시키기

In [6]:
!pip install wandb

In [7]:
import wandb
wandb.login()

True

In [ ]:
!pip install torchinfo

In [1]:
import torch

if torch.cuda.is_available():
    print("GPU 사용 가능")
    print("GPU 개수:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i} 이름:", torch.cuda.get_device_name(i))
        print(f"GPU {i} CUDA 버전:", torch.version.cuda)
else:
    print("GPU 사용 불가, CPU 사용 중")


GPU 사용 불가, CPU 사용 중


In [ ]:
import torch
from torch import nn, optim
from datetime import datetime
import os
from pathlib import Path

try:
  BASE_PATH = str(Path(__file__).resolve().parent.parent.parent)  # BASE_PATH: /Users/yhhan/git/link_dl
except NameError:
  BASE_PATH = str(Path.cwd())
import sys
sys.path.append('/content/link_dl')
os.listdir()

sys.path.append(BASE_PATH)

try:
  CURRENT_FILE_PATH = os.path.dirname(os.path.abspath(__file__))
except NameError:
  CURRENT_FILE_PATH = str(Path.cwd())
CHECKPOINT_FILE_PATH = os.path.join(CURRENT_FILE_PATH, "checkpoints")
os.makedirs(CHECKPOINT_FILE_PATH, exist_ok=True)
if not os.path.isdir(CHECKPOINT_FILE_PATH):
  os.makedirs(CHECKPOINT_FILE_PATH)

import sys
sys.path.append(BASE_PATH)

from _01_code._09_fcn_best_practice.c_trainer import ClassificationTrainer
from _03_homeworks.homework_3.a_fashion_mnist_data import get_fashion_mnist_data, get_fashion_mnist_test_data
from _01_code._16_modern_cnns.a_arg_parser import get_parser


def get_resnet_model(num_classes=10):
    class ResnetBlock(nn.Module):

      def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        self.downsample = downsample

      def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
          identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

    # ------------------------------------
    # ResNet-18
    # ------------------------------------
    class ResNet18(nn.Module):
      def __init__(self):
        super().__init__()

        # 처음 stem 부분 → Conv 사용
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=(3, 3), stride=(1, 1), padding=1, bias=False),
            nn.BatchNorm2d(num_features=32, eps=1e-05, momentum=0.1),
            nn.ReLU(inplace=True)
        )

        self.in_channels = 32

        # ResNet stages (2 blocks × 4 layers)
        # 1 → 32
        self.layer1 = self._make_layer(out_channels=32, blocks=2, stride=1)
        self.layer2 = self._make_layer(out_channels=64, blocks=2, stride=2)
        self.layer3 = self._make_layer(out_channels=128, blocks=2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=0.3)
        self.fc = nn.Linear(128, num_classes)

      def _make_layer(self, out_channels, blocks, stride):
        downsample = None

        # downsample 로직 → Conv/BatchNorm 활용
        if stride != 1:
          downsample = nn.Sequential(
              nn.Conv2d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
              nn.BatchNorm2d(out_channels)
        )

        layers = []
        layers.append(ResnetBlock(self.in_channels, out_channels, stride=stride, downsample=downsample))

        self.in_channels = out_channels

        for _ in range(1, blocks):
          layers.append(ResnetBlock(out_channels, out_channels))

        return nn.Sequential(*layers)

      def forward(self, x):
        x = self.stem(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

    # 1 * 28 * 28
    my_model = ResNet18()

    return my_model


def main(args):
  run_time_str = datetime.now().astimezone().strftime('%Y-%m-%d_%H-%M-%S')

  config = {
      'epochs': args.epochs,
      'batch_size': args.batch_size,
      'validation_intervals': args.validation_intervals,
      'learning_rate': args.learning_rate,
      'early_stop_patience': args.early_stop_patience,
      'early_stop_delta': args.early_stop_delta,
      'weight_decay': 5e-3
  }

  project_name = "cnn_fashion_mnist"
  name = "resnet_{0}".format(run_time_str)
  wandb.init(
      mode="online" if args.wandb else "disabled",
      project=project_name,
      notes="fashion mnist experiment with resnet",
      tags=["resnet", "fashion_mnist"],
      name=name,
      config=config
  )
  print(args)
  print(wandb.config)

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"Training on device {device}.")

  train_data_loader, validation_data_loader, fashion_mnist_transforms = get_fashion_mnist_data()
  model = get_resnet_model(num_classes=10)
  model.to(device)

  from torchinfo import summary
  summary(model=model,
          input_size=(1, 1, 28, 28),
          col_names=["kernel_size", "input_size", "output_size", "num_params", "mult_adds"]
  )

  optimizer = optim.SGD(
      model.parameters(),
      lr=wandb.config.learning_rate,
      momentum=0.9,
      weight_decay=config['weight_decay']
  )

  classification_trainer = ClassificationTrainer(
      project_name + "_resnet", model, optimizer, train_data_loader, validation_data_loader, fashion_mnist_transforms,
      run_time_str, wandb, device, CHECKPOINT_FILE_PATH
  )
  classification_trainer.train_loop()

  wandb.finish()

if __name__ == "__main__":
  # parser = get_parser()
  # args = parser.parse_args()
  # python _01_code/_09_modern_cnns/_02_googlenet/a_cifar10_train_googlenet.py --wandb -v 10
  from types import SimpleNamespace

  args = SimpleNamespace(
      wandb=True,
      batch_size=1024,
      epochs=500,
      learning_rate=0.005,
      validation_intervals=10,
      early_stop_patience=10,
      early_stop_delta=1e-5,
      weight_decay=0.005
  )

  main(args)

Namespace(wandb=False, batch_size=2048, epochs=500, learning_rate=0.001, validation_intervals=10, early_stop_patience=10, early_stop_delta=1e-05)
{'epochs': 500, 'batch_size': 2048, 'validation_intervals': 10, 'learning_rate': 0.001, 'early_stop_patience': 10, 'early_stop_delta': 1e-05, 'weight_decay': 0.0001}
Training on device cpu.
Num Train Samples:  55000
Num Validation Samples:  5000
Sample Data Shape:  torch.Size([1, 28, 28])
Sample Data Target:  1
Number of Data Loading Workers: 16


C:\Users\WJM\anaconda3\envs\link_dl\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[Epoch   1] T_loss: 0.85535, T_accuracy: 71.4582 | V_loss: 1.09089, V_accuracy: 62.5200 | Early stopping is stated! | T_time: 00:16:25, T_speed: 0.001
[Epoch  10] T_loss: 0.11880, T_accuracy: 95.9255 | V_loss: 0.47431, V_accuracy: 85.9000 | V_loss decreased (1.09089 --> 0.47431). Saving model... | T_time: 02:45:12, T_speed: 0.001
[Epoch  20] T_loss: 0.01616, T_accuracy: 99.6236 | V_loss: 0.49613, V_accuracy: 89.6000 | Early stopping counter: 1 out of 10 | T_time: 05:06:49, T_speed: 0.001
